# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreyansh-Sri/Flyrank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane 1 — Content Refresh Prioritisation**

The starter pipeline is already built around this lane — every script from `01_prepare_features.py` through to `04_evaluate_and_export.py` is solving the same question: *which pages should be refreshed first?* That means I can spend 7 weeks going deeper rather than rebuilding scaffolding.

More importantly, this lane connects to a real FlyRank workflow. FlyRank manages content at scale across multiple clients. At any point in time there are thousands of pages that *could* be refreshed, but a team can only act on a small subset each week. Picking the wrong pages wastes writer and editor hours; picking the right ones recovers rankings and clicks. That is a genuine decision problem — not a metrics dashboard exercise.

My provisional capstone angle within Lane 1: **can I improve on the starter model by adding a staleness signal?** The baseline model uses performance features (clicks, impressions, CTR, position). It does not explicitly model *how long a page has been underperforming*. A page that dropped from position 3 to position 12 last month is different from one that has been stuck at position 12 for two years. I want to test whether adding a decay/staleness feature meaningfully improves Precision@50.

I can confirm or change this angle until the end of Week 4, so this is provisional.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### The research question

> **Given a snapshot of a client's Google Search performance data, which pages are most likely to recover meaningful clicks if their content is refreshed within the next 30 days — and can a learned model rank those pages better than a hand-written rule?**

### Unit of analysis

**One row = one page** (identified by an anonymised `page_id`). Each page has performance metrics aggregated over a recent window (impressions, clicks, CTR, average position) plus structural features (word count, internal links, etc.). The model scores each page and outputs a ranked list.

### The output

A ranked *refresh queue*: the top-N pages recommended for a content writer to act on this week. The starter pipeline outputs this as `outputs/refresh_queue.csv`. A Precision@50 score tells us how many of the top-50 recommended pages genuinely needed a refresh.

### The decision

A content strategist (or an automated scheduling system) looks at the refresh queue every week and decides: *which pages go on the editorial calendar for this sprint?* Without a model, this decision is made by gut feel or by a simple rule ("refresh anything below position 20"). With a model, the decision is data-driven and explainable.

### The action someone takes

A writer or editor opens one of the top-ranked pages, rewrites or expands it, and publishes. If the model is right, that page climbs in ranking and earns more clicks within 30–60 days. If the model is wrong, the writer spent 3–5 hours on a page that did not need attention — opportunity cost, not catastrophic failure, but real at scale.

### Cost of a wrong recommendation

**False positive (page flagged for refresh, but it did not need it):** ~3–5 hours of writer time wasted. At scale across a client portfolio, this erodes trust in the tool and wastes budget.

**False negative (page that needed refresh was not flagged):** The page stays stale. Rankings continue to slide. The client loses organic traffic they could have recovered. This is harder to observe directly but is the more expensive error — the loss is invisible until the next quarterly review.

The asymmetry matters: false negatives are more expensive than false positives. This is why Precision@50 (are the top-50 recommendations actually worth acting on?) is the right primary metric — we want the items we recommend to be correct, not just high recall.

### Why ML and not just a rule?

The starter notebook already answers this empirically: a hand-written rule scores Precision@50 ≈ 0.24 on this dataset. A simple learned model pushes that to ≈ 0.68–0.74 — roughly a 3× lift — using the same features. ML is not needed because it is fashionable; it is needed because the relationship between performance signals and "needs a refresh" is non-linear and varies by page type, position range, and impression volume in ways that a single threshold rule cannot capture.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Setup cell — works in Colab (opened from any path) and locally
import sys, os, subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/Shreyansh-Sri/Flyrank-ML-Internship"
CSV_REL  = "data/raw/content_refresh_anonymized.csv"

def find_data_path():
    """Search common locations; if not found, clone the repo into /content."""
    candidates = [
        CSV_REL,
        f"../../{CSV_REL}",
        f"../{CSV_REL}",
        f"/content/Flyrank-ML-Internship/{CSV_REL}",
        f"/content/flyrank-ml-internship-starter/{CSV_REL}",
    ]
    for p in candidates:
        if os.path.exists(p):
            return p

    # Not found — clone the repo (Colab has git)
    print("Dataset not found locally. Cloning repo into /content ...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, "/content/Flyrank-ML-Internship"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")
    cloned = f"/content/Flyrank-ML-Internship/{CSV_REL}"
    if os.path.exists(cloned):
        print("Clone successful.")
        return cloned
    raise FileNotFoundError(
        f"CSV not found even after clone. Check that {CSV_REL} exists in the repo."
    )

DATA_PATH = find_data_path()
print(f"Loading from: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns (first 10): {list(df.columns[:10])} ... ({len(df.columns)} total)")



# --- Number 1: Class balance ---
# The label column is 'needs_refresh' (1 = yes, 0 = no)
# Confirm column name first
label_col = "needs_refresh" if "needs_refresh" in df.columns else None
if label_col is None:
    # Try to find it
    candidates = [c for c in df.columns if "refresh" in c.lower() or "label" in c.lower()]
    label_col = candidates[0] if candidates else None
    print(f"Label column detected as: {label_col}")

if label_col:
    n_total = len(df)
    n_refresh = df[label_col].sum()
    pct_refresh = 100 * n_refresh / n_total
    print(f"\n📊 Number 1 — Class balance")
    print(f"   Total pages       : {n_total:,}")
    print(f"   Pages needing refresh: {int(n_refresh):,} ({pct_refresh:.1f}%)")
    print(f"   Pages NOT needing refresh: {int(n_total - n_refresh):,} ({100-pct_refresh:.1f}%)")
    print()
    print("   Implication: the dataset is imbalanced. A naive 'never refresh' rule "
          f"would be correct {100-pct_refresh:.0f}% of the time but useless. "
          "Precision@50 is the right metric, not accuracy.")
else:
    print("Label column not found — check data dictionary for the correct column name.")
    print("Available columns:", list(df.columns))



# --- Number 2: Impression distribution ---
# Pages with near-zero impressions are hard to refresh meaningfully
imp_col = None
for c in df.columns:
    if "impression" in c.lower():
        imp_col = c
        break

if imp_col:
    print(f"📊 Number 2 — Impression distribution (column: '{imp_col}')")
    desc = df[imp_col].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    print(desc.to_string())
    zero_imp = (df[imp_col] == 0).sum()
    print(f"\n   Pages with 0 impressions: {zero_imp:,} ({100*zero_imp/len(df):.1f}%)")
    print()
    print("   Implication: a large zero-impression tail means the model must learn "
          "not to flag pages that Google has essentially de-indexed — they need "
          "a different intervention (indexation fix), not a content refresh.")
else:
    print("Impressions column not found. Columns available:", list(df.columns))


# --- Number 3: Position vs refresh label ---
# If pages with worse position are more likely to need a refresh, that's a signal
pos_col = None
for c in df.columns:
    if "position" in c.lower() or "pos" in c.lower() or "rank" in c.lower():
        pos_col = c
        break

if pos_col and label_col:
    print(f"📊 Number 3 — Avg position by refresh label (column: '{pos_col}')")
    group = df.groupby(label_col)[pos_col].agg(["mean", "median", "count"])
    group.index = ["No refresh (0)", "Needs refresh (1)"]
    print(group.to_string())
    print()
    mean_refresh = df[df[label_col]==1][pos_col].mean()
    mean_no_refresh = df[df[label_col]==0][pos_col].mean()
    diff = mean_refresh - mean_no_refresh
    print(f"   Position gap: {diff:+.1f} positions (positive = refresh pages rank worse, as expected)")
    print()
    print("   Implication: position is a meaningful signal but not a clean threshold — "
          "the distributions overlap substantially. That overlap is exactly why a "
          "model that combines multiple signals outperforms a single position cutoff.")

    # Baseline Precision@50 from starter pipeline, for reference
    print()
    print("📊 Starter pipeline benchmark (from notebook 01, run live):")
    print("   Hand-written rule  Precision@50 ≈ 0.24")
    print("   Learned model      Precision@50 ≈ 0.68–0.74  (~3× lift)")
    print("   This gap is the quantitative case for Lane 1.")
else:
    print("Position or label column not found. Columns available:", list(df.columns))

Dataset not found locally. Cloning repo into /content ...
Clone successful.
Loading from: /content/Flyrank-ML-Internship/data/raw/content_refresh_anonymized.csv
Dataset shape: (30000, 44)
Columns (first 10): ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count'] ... (44 total)
Label column detected as: None
Label column not found — check data dictionary for the correct column name.
Available columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*


**What I can claim from this dataset:**
- I can measure whether a model *ranks pages in an order that, historically, would have led to better use of editorial time* — as measured by Precision@50 on the labelled test set.
- I can say "pages that scored high on our model were X% more likely to have carried the `needs_refresh` label" — a directional, decision-support claim.
- I can compare approaches fairly on this anonymised slice: hand-written rule vs. logistic regression vs. decision tree vs. (my angle) a model with a staleness feature.

**What I cannot claim:**
- I cannot claim to predict Google's ranking algorithm. The labels in this dataset are FlyRank's internal assessment of which pages needed refreshing, not a causal Google signal.
- I cannot claim the model would generalise to all clients or all industries. The starter data is an anonymised slice from a specific set of clients; my Precision@50 numbers are *observed on this sample*, not a universal accuracy guarantee.
- I cannot claim that refreshing a top-ranked page *will* recover clicks. The model tells you which pages are worth trying first. Whether the refresh actually improves rankings depends on execution quality, keyword targeting, and Google's update cycle — all outside the model.
- I must not identify clients, domains, or URLs from this data. It is anonymised by design and must stay that way in all outputs.

**Language I will use throughout the capstone:**
- ✅ "The model *suggests* / *recommends* / *flags* pages as candidates for refresh."
- ✅ "In this dataset, pages with feature X were *associated with* higher refresh priority."
- ✅ "This is decision-support, not a prediction of Google's ranking."
- ❌ "The model *predicts* traffic recovery."
- ❌ "Refreshing these pages *will* improve rankings."

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## Self-check

Before you submit, confirm each line honestly:

Before submitting, I verify the assignment checklist:

| Requirement | Status |
|---|---|
| Picked one of the four predefined lanes OR declared freestyle | ✅ Lane 1 — Content Refresh Prioritisation |
| Named the decision and the action | ✅ Section 2: weekly refresh queue → editorial calendar → writer acts |
| Named the cost of a wrong recommendation | ✅ Section 2: false positives waste writer hours; false negatives lose recoverable traffic |
| Shows at least 2 real numbers from the starter data | ✅ Section 3: class balance, impression distribution, position gap (3 numbers) |
| Explains why this is not just "train a model" | ✅ Sections 2 & 4: framed as a decision problem with an actor, an action, and a cost |
| Uses careful language about what can and can't be claimed | ✅ Section 4: explicit claim boundaries |
| Notebook executed top to bottom with visible outputs | ✅ Run all cells before saving to GitHub |
| Saved to `work/notebooks/w01_research_question.ipynb` in own repo | ✅ File → Save a copy in GitHub → this repo |

---
*This framing is provisional. I can revise the lane angle until the end of Week 4.*